Since fsps has compile-time options in order to change the spectral libraries and isochrones, 
we have to install the package using the command

``
FFLAGS="-DC3K_LR=0 -DC3K_HR=1" pip install fsps --no-binary fsps
``

This enables the higher resolution BaSeL spectral library.

In [ ]:
import numpy as np

In [ ]:
import fsps

In [ ]:
from astropy.constants import c

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
C_KMS = c.to("km/s").value

In [ ]:
C_KMS

In [ ]:
sp = fsps.StellarPopulation(compute_vega_mags=False, 
                            zcontinuous=0, # use all metallicities
                            sfh=0, logzsol=0.0, dust_type=1, )

In [ ]:
sp.spec_library

In [ ]:
sp.isoc_library

In [ ]:
ages = 10**(sp.ssp_ages - 9)

In [ ]:
metals = np.log10(sp.zlegend / 0.016)

In [ ]:
waves = sp.wavelengths

In [ ]:
templates = np.empty((len(waves), len(ages), len(metals)))

In [ ]:
for i in range(len(metals)):
    wave, specs = sp.get_spectrum(zmet=i + 1)
    templates[:, :, i] = specs.T

In [ ]:
masses = np.empty((len(ages), len(metals)))

In [ ]:
for i in range(len(metals)):
    sp.params["zmet"] = i+1
    masses[:, i] = sp.stellar_mass

In [ ]:
sigma_A = np.abs(sp.resolutions) / C_KMS * waves

In [ ]:
fwhm = 2*np.sqrt(2*np.log(2)) * sigma_A

In [ ]:
np.savez("../sps_templates/spectra_fsps_me.npz", templates=templates, masses=masses, lam=lam, ages=ages, metals=metals, fwhm=fwhm)


# New

In [ ]:
import arya

In [ ]:
arya.init("mnras")

In [ ]:
templates.shape

In [ ]:
hm = arya.HueMap((metals[0], metals[-1]))
dy = 2e-16
for i in range(len(metals)):
    p = plt.plot(lam, templates[:, -1, i] + i*dy, color=hm(metals[i]), lw=0.5)
    plt.xlim(3000, 10_000)


arya.Colorbar(hm, ax=plt.gca(), label="[Fe/H]")
plt.xlabel("wavelength / $\\AA$")
plt.ylabel("flux (arbitrary)")

## testing
what is the structure of the other file

In [ ]:
df = np.load("../sps_templates/spectra_fsps_9.0.npz")

In [ ]:
list(df.keys())

In [ ]:
df["templates"].shape

In [ ]:
df["ages"].shape

In [ ]:
df["masses"].shape